# M3L3 E02 — Agentes especialistas con RAG (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- separar índices y agentes por dominio.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Índices separados

Un especialista tiene su propio conocimiento. En M3L2 lo haríamos con `FAISS.from_texts`; acá usamos `retrieve()` para que la idea corra sin dependencias.

```text
query -> HRAgent -> índice HR
query -> TechAgent -> índice Tech
query -> BillingAgent -> índice Billing
```


Creamos documentos por dominio, palabras clave y un retriever simple que devuelve los textos más relevantes.


In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

KEYWORDS = {
    "hr": ["vacaciones", "beneficio", "seguro", "licencia"],
    "tech": ["vpn", "contraseña", "mfa", "notebook"],
    "billing": ["factura", "facturas", "reembolso", "pago", "recibo"],
}

def detect_domains(query: str) -> list[str]:
    text = query.lower()
    matches = [domain for domain, words in KEYWORDS.items() if any(word in text for word in words)]
    return matches or ["unknown"]

def retrieve(domain: str, query: str, k: int = 2) -> list[str]:
    docs = knowledge_base[domain]
    query_words = set(query.lower().replace("¿", "").replace("?", "").split())
    def score(doc: str) -> int:
        return sum(1 for word in query_words if word.strip(",.") in doc.lower())
    return sorted(docs, key=score, reverse=True)[:k]


## Sección 2 — Encapsular agentes

`make_agent(domain, display_name)` devuelve una función que recibe `query: str` y responde usando solo ese dominio.


In [ ]:
def make_agent(domain: str, display_name: str) -> Callable[[str], str]:
    def agent(query: str) -> str:
        return f"{display_name}: " + " | ".join(retrieve(domain, query, k=2))
    return agent

hr_agent = make_agent("hr", "HRAgent")
tech_agent = make_agent("tech", "TechAgent")
billing_agent = make_agent("billing", "BillingAgent")


## Sección 3 — Índice correcto vs incorrecto

La misma consulta de vacaciones produce mejor contexto en HR que en Tech. Esa comparación vuelve visible el valor de separar índices.


In [ ]:
print(hr_agent("¿Cómo solicito vacaciones?"))
print(tech_agent("¿Cómo solicito vacaciones?"))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert "HRAgent" in hr_agent("vacaciones")
    assert "TechAgent" in tech_agent("vpn")
    print("Checks E02 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Separar índices y agentes por dominio.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
